In [1]:
import numpy as np
import pandas as pd

from itertools import product
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [6]:
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "engineered.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,school_district_Williams Unified,school_district_Willits Unified,school_district_Willows Unified,school_district_Windsor Unified,school_district_Wiseburn Unified,school_district_Woodlake Unified,school_district_Woodland Joint Unified,school_district_Yosemite Unified,school_district_Yuba City Unified,school_district_Yucaipa-Calimesa Joint Unified
0,890000.0,202506,3000.0,181,9600.0,2021.0,3.0,3.0,2.0,34.264692,...,0,0,0,0,0,0,0,0,0,0
1,1876384.0,202506,1800.0,87,10400.0,1963.0,3.0,3.0,2.0,34.107983,...,0,0,0,0,0,0,0,0,0,0
2,4820000.0,202506,4270.0,0,22505.0,1980.0,6.0,6.0,3.0,37.567434,...,0,0,0,0,0,0,0,0,0,0
3,865000.0,202506,1442.0,0,4800.0,1985.0,2.0,3.0,2.0,33.906058,...,0,0,0,0,0,0,0,0,0,0
4,875000.0,202506,1086.0,0,5500.0,1953.0,1.0,3.0,4.0,37.705919,...,0,0,0,0,0,0,0,0,0,0


In [7]:
target = "ClosePrice"

drop_cols = [
    target,
    "source_month"
]

feature_cols = [
    column
    for column in df.columns
    if column not in drop_cols
]

months = sorted(
    df["source_month"].unique()
)

print("Number of features:", len(feature_cols))
print("Available months:", months)
print("Final test month:", months[-1])

Number of features: 392
Available months: [202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605, 202606]
Final test month: 202606


In [8]:
parameter_grid = list(
    product(
        [3, 5],            # max_depth
        [0.05, 0.10],      # learning_rate
        [200, 400, 600]    # n_estimators
    )
)

print(
    "Parameter combinations:",
    len(parameter_grid)
)

Parameter combinations: 12


In [10]:
advanced_results = []
tuning_results = []
trained_models = {}

test_month = months[-1]


for X_window in [3, 6, 9, 12]:

    train_months = months[-(X_window + 1):-1]

    validation_month = train_months[-1]
    fitting_months = train_months[:-1]


    fitting_df = df[
        df["source_month"].isin(
            fitting_months
        )
    ].copy()

    validation_df = df[
        df["source_month"] == validation_month
    ].copy()

    full_train_df = df[
        df["source_month"].isin(
            train_months
        )
    ].copy()

    test_df = df[
        df["source_month"] == test_month
    ].copy()


    X_fit = fitting_df[feature_cols].copy()
    y_fit = fitting_df[target].copy()

    X_validation = validation_df[
        feature_cols
    ].copy()

    y_validation = validation_df[
        target
    ].copy()

    X_train = full_train_df[
        feature_cols
    ].copy()

    y_train = full_train_df[
        target
    ].copy()

    X_test = test_df[
        feature_cols
    ].copy()

    y_test = test_df[
        target
    ].copy()


    print(
        f"\n{X_window}-month window"
    )

    print(
        "Fitting months:",
        fitting_months
    )

    print(
        "Validation month:",
        validation_month
    )

    print(
        "Test month:",
        test_month
    )


    best_validation_rmse = np.inf
    best_parameters = None


    for (
        max_depth,
        learning_rate,
        n_estimators
    ) in parameter_grid:

        model = XGBRegressor(
            objective="reg:squarederror",
            max_depth=max_depth,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_fit,
            y_fit
        )

        validation_predictions = model.predict(
            X_validation
        )

        validation_mae = mean_absolute_error(
            y_validation,
            validation_predictions
        )

        validation_rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                validation_predictions
            )
        )

        validation_r2 = r2_score(
            y_validation,
            validation_predictions
        )


        tuning_results.append(
            {
                "training_window_months": X_window,
                "validation_month": validation_month,
                "max_depth": max_depth,
                "learning_rate": learning_rate,
                "n_estimators": n_estimators,
                "validation_mae": validation_mae,
                "validation_rmse": validation_rmse,
                "validation_r2": validation_r2
            }
        )


        if validation_rmse < best_validation_rmse:

            best_validation_rmse = validation_rmse

            best_parameters = {
                "max_depth": max_depth,
                "learning_rate": learning_rate,
                "n_estimators": n_estimators
            }


    final_model = XGBRegressor(
        objective="reg:squarederror",
        max_depth=best_parameters[
            "max_depth"
        ],
        learning_rate=best_parameters[
            "learning_rate"
        ],
        n_estimators=best_parameters[
            "n_estimators"
        ],
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(
        X_train,
        y_train
    )

    trained_models[X_window] = final_model


    train_predictions = final_model.predict(
        X_train
    )

    test_predictions = final_model.predict(
        X_test
    )


    advanced_results.append(
    {
        "model": "XGBoost",
        "training_window_months": X_window,
        "train_months": ", ".join(
            map(str, train_months)
        ),
        "validation_month": str(validation_month),
        "test_month": str(test_month),
        "max_depth": best_parameters["max_depth"],
        "learning_rate": best_parameters["learning_rate"],
        "n_estimators": best_parameters["n_estimators"],
        "train_r2": r2_score(
            y_train,
            train_predictions
        ),
        "test_r2": r2_score(
            y_test,
            test_predictions
        ),
        "mae": mean_absolute_error(
            y_test,
            test_predictions
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_test,
                test_predictions
            )
        )
    }
)


    


3-month window
Fitting months: [202603, 202604]
Validation month: 202605
Test month: 202606

6-month window
Fitting months: [202512, 202601, 202602, 202603, 202604]
Validation month: 202605
Test month: 202606

9-month window
Fitting months: [202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604]
Validation month: 202605
Test month: 202606

12-month window
Fitting months: [202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604]
Validation month: 202605
Test month: 202606


In [11]:
advanced_results_df = pd.DataFrame(
    advanced_results
)

advanced_results_df = (
    advanced_results_df
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .reset_index(drop=True)
)

advanced_results_df

,model,training_window_months,train_months,validation_month,test_month,max_depth,learning_rate,n_estimators,train_r2,test_r2,mae,rmse
0,XGBoost,12,"202506, 202507, 202508, 202509, 202510, 202511...",202605,202606,3,0.05,200,0.247697,0.484318,435269.853813,1.103416e+06
1,XGBoost,9,"202509, 202510, 202511, 202512, 202601, 202602...",202605,202606,3,0.05,200,0.243562,0.392017,442408.019786,1.198102e+06
2,XGBoost,6,"202512, 202601, 202602, 202603, 202604, 202605",202605,202606,3,0.05,200,0.463654,0.050773,459026.204381,1.497039e+06
3,XGBoost,3,"202603, 202604, 202605",202605,202606,3,0.05,200,0.759593,-0.874438,496778.222835,2.103699e+06


In [14]:
tuning_results_df = pd.DataFrame(
    tuning_results
)

tuning_results_df = (
    tuning_results_df
    .sort_values(
        by=[
            "training_window_months",
            "validation_rmse"
        ]
    )
    .reset_index(drop=True)
)

tuning_results_df

,training_window_months,validation_month,max_depth,learning_rate,n_estimators,validation_mae,validation_rmse,validation_r2
0,3,202605,3,0.05,200,616501.195655,6.239779e+06,-12.814994
1,3,202605,5,0.10,200,490677.531355,6.528594e+06,-14.123473
2,3,202605,5,0.10,400,462902.740813,6.603722e+06,-14.473545
3,3,202605,5,0.10,600,444145.334628,6.605559e+06,-14.482152
4,3,202605,5,0.05,200,506851.357912,6.631123e+06,-14.602222
5,3,202605,3,0.05,400,599065.059522,6.834800e+06,-15.575393
6,3,202605,5,0.05,400,478576.203390,6.842167e+06,-15.611143
7,3,202605,5,0.05,600,458901.517492,6.886179e+06,-15.825535
8,3,202605,3,0.05,600,595894.504633,7.081971e+06,-16.795924
9,3,202605,3,0.10,200,616889.749952,7.212767e+06,-17.459337


In [13]:
best_window = int(
    advanced_results_df.loc[
        0,
        "training_window_months"
    ]
)

best_model = trained_models[
    best_window
]

feature_importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": (
            best_model.feature_importances_
        )
    }
)

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_df.head(15)

,feature,importance
0,MainLevelBedrooms,0.215930
1,living_area_per_bathroom,0.104150
2,DaysOnMarket,0.073827
3,BedroomsTotal,0.052479
4,total_bed_bath,0.052222
5,LotSizeSquareFeet,0.050505
6,GarageSpaces,0.046084
7,bed_bath_ratio,0.042687
8,lot_to_living_ratio,0.035886
9,BathroomsTotalInteger,0.029041


In [15]:
advanced_results_df.to_csv(
    folder / "xgboost_test_metrics.csv",
    index=False
)

tuning_results_df.to_csv(
    folder / "xgboost_tuning_results.csv",
    index=False
)

feature_importance_df.to_csv(
    folder / "xgboost_feature_importance.csv",
    index=False
)